# MediGuide — LoRA Fine-Tuning
### Qwen2.5-1.5B-Instruct + PEFT LoRA on Doctor-Patient Conversations

Fine-tunes the base model from the baseline notebook using LoRA (attention + MLP target
modules), then re-runs the **exact same evaluation pipeline** (ROUGE-L, BLEU, perplexity,
latency) used in the baseline notebook, and diffs the two.

**Environment:** Kaggle, single T4 (16GB), fp16 (T4 has no native bf16 tensor-core support).

**Carried over from the baseline run, unchanged:**
- Stock Qwen2.5 tokenizer, **no** `add_tokens` / `resize_token_embeddings` — this is what caused
  the 905MB adapter and the `151665` vs `151936` vocab mismatch last time.
- `seed = 8` everywhere.
- Same prompt template (system + user → assistant).


## 1. Setup

In [ ]:
!pip install -q -U peft accelerate bitsandbytes evaluate rouge_score sacrebleu sentencepiece


In [ ]:
import os, json, time, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

SEED = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Torch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
DATA_DIR = "/kaggle/input/datasets/totaldose/doctor-patient-conversation/data/processed"

def find_split_file(data_dir, split_names):
    for name in split_names:
        p = os.path.join(data_dir, f"{name}.json")
        if os.path.exists(p):
            return p
    return None

TRAIN_PATH = find_split_file(DATA_DIR, ["train"])
VAL_PATH   = find_split_file(DATA_DIR, ["val", "valid", "validation", "dev"])
TEST_PATH  = find_split_file(DATA_DIR, ["test"])

print("train:", TRAIN_PATH)
print("val:  ", VAL_PATH)
print("test: ", TEST_PATH)
assert TRAIN_PATH is not None and TEST_PATH is not None, "Check DATA_DIR"


In [ ]:
# Where to find the baseline results saved by the previous notebook, for the final diff.
# Adjust if this notebook is running in a different Kaggle session than the baseline one.
BASELINE_SUMMARY_CANDIDATES = [
    "/kaggle/working/results/baseline_summary.json",
    "/kaggle/input/mediguide-baseline/baseline_summary.json",
]
BASELINE_SUMMARY_PATH = next((p for p in BASELINE_SUMMARY_CANDIDATES if os.path.exists(p)), None)
print("Baseline summary found at:", BASELINE_SUMMARY_PATH)
if BASELINE_SUMMARY_PATH is None:
    print("WARNING: baseline_summary.json not found automatically — set BASELINE_SUMMARY_PATH "
          "manually, or upload it as a Kaggle dataset, to get the comparison table at the end.")


## 2. Tokenizer — unmodified, sanity-checked

Same check as the baseline notebook. We never call `add_tokens` or `resize_token_embeddings`
anywhere in this notebook.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
config = AutoConfig.from_pretrained(MODEL_NAME)

print("len(tokenizer):    ", len(tokenizer))
print("config.vocab_size: ", config.vocab_size)
assert config.vocab_size >= len(tokenizer)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # right-padding for training (left-padding is only for batched generation)


## 3. Load base model + apply LoRA

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float16
).to(DEVICE)

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False  # required when using gradient checkpointing

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # attention + MLP
    bias="none",
    # deliberately NOT setting modules_to_save — embed_tokens/lm_head stay frozen since we
    # never touch the tokenizer/vocab. This is what keeps the adapter small this time.
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


## 4. Dataset preparation

Same prompt template as the baseline notebook: system message (clinical/professional/disclaimer
instructions) + user turn (`Description` + `Patient`) → assistant turn (`Doctor`).

For training we build the **full** sequence `prompt_tokens + target_tokens + eos`, and mask the
prompt portion of the labels with `-100` so loss is only computed over the `Doctor` response —
the model isn't penalized for "predicting" the patient's own question back.

`MAX_PROMPT_LENGTH` is raised to 768 here (baseline eval used 512, which silently truncated 2/333
very long `Patient` messages mid-sentence). Doing it right here avoids that during training.


In [ ]:
SYSTEM_PROMPT = (
    "You are a medical information assistant. Respond to the patient's question with "
    "clear, professional, and clinically sound guidance, consistent with recognized clinical "
    "guidelines. Use formal medical language appropriate for a patient audience. Always make "
    "clear that your response is informational only and does not replace an in-person "
    "diagnosis or professional medical care."
)

MAX_PROMPT_LENGTH = 768
MAX_TARGET_LENGTH = 512
MAX_TOTAL_LENGTH = MAX_PROMPT_LENGTH + MAX_TARGET_LENGTH

def build_prompt(description, patient, for_generation=True):
    user_turn = f"{description.strip()}\n\n{patient.strip()}"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_turn},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_generation
    )

def encode_example(description, patient, doctor):
    prompt_text = build_prompt(description, patient, for_generation=True)
    prompt_ids = tokenizer(
        prompt_text, truncation=True, max_length=MAX_PROMPT_LENGTH, add_special_tokens=False
    )["input_ids"]
    target_ids = tokenizer(
        doctor.strip() + tokenizer.eos_token,
        truncation=True, max_length=MAX_TARGET_LENGTH, add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


In [ ]:
class DoctorPatientDataset(Dataset):
    def __init__(self, df):
        self.rows = df.reset_index(drop=True)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows.iloc[idx]
        return encode_example(row["Description"], row["Patient"], row["Doctor"])


In [ ]:
def load_split(path):
    with open(path) as f:
        return pd.DataFrame(json.load(f))

df_train = load_split(TRAIN_PATH)
df_val = load_split(VAL_PATH) if VAL_PATH else None
df_test = load_split(TEST_PATH)

print("train:", len(df_train))
if df_val is not None:
    print("val:  ", len(df_val))
print("test: ", len(df_test))

train_dataset = DoctorPatientDataset(df_train)
eval_dataset = DoctorPatientDataset(df_val) if df_val is not None else None

# Sanity-check one encoded example
sample = train_dataset[0]
print("\nEncoded example — total length:", len(sample["input_ids"]),
      "| masked (prompt) tokens:", sample["labels"].count(-100))


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)


## 5. Training configuration

Batch size kept small with gradient accumulation to stay within T4's 16GB alongside gradient
checkpointing. `fp16=True` (not bf16 — T4 doesn't have hardware bf16 support). 3 epochs is a
reasonable starting point for ~2.6K training examples on a 1.5B model with LoRA; watch the
eval-loss curve and reduce if it starts overfitting before epoch 3.


In [ ]:
OUTPUT_DIR = "/kaggle/working/lora_checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,   # effective batch size 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch" if eval_dataset is not None else "no",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=eval_dataset is not None,
    metric_for_best_model="eval_loss" if eval_dataset is not None else None,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)


In [ ]:
train_result = trainer.train()
print(train_result)


## 6. Save the adapter — and confirm it's small this time

This is the direct check against last run's bug: adapter size should be in the tens of MB, and
`adapter_config.json` should show `modules_to_save: null` (i.e. `embed_tokens`/`lm_head` were
never touched).


In [ ]:
ADAPTER_DIR = "/kaggle/working/lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_file = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
if os.path.exists(adapter_file):
    size_mb = os.path.getsize(adapter_file) / (1024 * 1024)
    print(f"Adapter size: {size_mb:.1f} MB")
else:
    print("adapter_model.safetensors not found — check ADAPTER_DIR contents:", os.listdir(ADAPTER_DIR))

with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    adapter_cfg = json.load(f)
print("modules_to_save:", adapter_cfg.get("modules_to_save"))
assert not adapter_cfg.get("modules_to_save"), \
    "modules_to_save is set — embed_tokens/lm_head got included again, check tokenizer usage above."


## 7. Re-run the baseline evaluation pipeline on the LoRA model

Same generation settings, same perplexity scoring, same metrics as the baseline notebook, so the
comparison at the end is apples-to-apples. `model` here is already the PEFT-wrapped model from
training (adapter active).


In [ ]:
model.eval()
model.config.use_cache = True  # re-enable KV cache for generation (was off for training)
tokenizer.padding_side = "left"  # required for batched generation, opposite of training

GEN_MAX_NEW_TOKENS = 256   # matches baseline notebook
GEN_BATCH_SIZE = 8

def generate_batch(prompts, max_new_tokens=GEN_MAX_NEW_TOKENS):
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_PROMPT_LENGTH
    ).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    elapsed = time.time() - t0
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return [t.strip() for t in texts], elapsed / len(prompts)


In [ ]:
prompts = [build_prompt(row["Description"], row["Patient"]) for _, row in df_test.iterrows()]

predictions = []
gen_seconds = []

t0 = time.time()
for i in range(0, len(prompts), GEN_BATCH_SIZE):
    batch = prompts[i:i + GEN_BATCH_SIZE]
    texts, per_ex_time = generate_batch(batch)
    predictions.extend(texts)
    gen_seconds.extend([per_ex_time] * len(texts))
    print(f"{min(i + GEN_BATCH_SIZE, len(prompts))}/{len(prompts)} done, "
          f"{time.time() - t0:.1f}s elapsed", flush=True)

df_test["prediction"] = predictions
df_test["gen_seconds"] = gen_seconds
total_gen_time = time.time() - t0
print(f"Total generation time: {total_gen_time:.1f}s")


In [ ]:
def compute_example_loss(prompt, target):
    prompt_ids = tokenizer(prompt, truncation=True, max_length=MAX_PROMPT_LENGTH,
                            add_special_tokens=False)["input_ids"]
    target_ids = tokenizer(target + tokenizer.eos_token, add_special_tokens=False,
                            truncation=True, max_length=MAX_TARGET_LENGTH)["input_ids"]
    input_ids = torch.tensor([prompt_ids + target_ids]).to(DEVICE)
    labels = input_ids.clone()
    labels[:, :len(prompt_ids)] = -100

    with torch.no_grad():
        out = model(input_ids=input_ids, labels=labels)
    return out.loss.item(), len(target_ids)


In [ ]:
losses, n_target_tokens = [], []
t0 = time.time()
for i, row in df_test.iterrows():
    prompt = build_prompt(row["Description"], row["Patient"])
    loss, n_tok = compute_example_loss(prompt, row["Doctor"])
    losses.append(loss)
    n_target_tokens.append(n_tok)
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(df_test)} scored, {time.time() - t0:.1f}s elapsed", flush=True)

df_test["loss"] = losses
df_test["n_target_tokens"] = n_target_tokens

total_loss_tokens = (df_test["loss"] * df_test["n_target_tokens"]).sum()
total_tokens = df_test["n_target_tokens"].sum()
corpus_ppl = math.exp(total_loss_tokens / total_tokens)
df_test["perplexity"] = df_test["loss"].apply(math.exp)
print(f"Corpus-level perplexity (LoRA fine-tuned): {corpus_ppl:.3f}")


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

references = df_test["Doctor"].tolist()
rouge_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
bleu_scores = bleu.compute(predictions=predictions, references=[[r] for r in references])

rouge_per_row = [rouge.compute(predictions=[p], references=[r], use_stemmer=True)["rougeL"]
                  for p, r in zip(predictions, references)]
df_test["rougeL"] = rouge_per_row

print("ROUGE:", rouge_scores)
print("BLEU: ", bleu_scores["score"])


## 8. LoRA results summary + diff against baseline

In [ ]:
lora_summary = {
    "model": MODEL_NAME + " + LoRA",
    "seed": SEED,
    "n_test_examples": len(df_test),
    "rouge1": rouge_scores["rouge1"],
    "rouge2": rouge_scores["rouge2"],
    "rougeL": rouge_scores["rougeL"],
    "bleu": bleu_scores["score"],
    "perplexity_corpus": corpus_ppl,
    "avg_generation_seconds_per_example": df_test["gen_seconds"].mean(),
    "max_new_tokens": GEN_MAX_NEW_TOKENS,
    "total_wall_clock_generation_seconds": total_gen_time,
    "adapter_size_mb": size_mb if 'size_mb' in dir() else None,
}

os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/lora_summary.json", "w") as f:
    json.dump(lora_summary, f, indent=2)
df_test.to_csv("/kaggle/working/results/lora_predictions.csv", index=False)

pd.DataFrame([lora_summary]).T.rename(columns={0: "value"})


In [ ]:
if BASELINE_SUMMARY_PATH:
    with open(BASELINE_SUMMARY_PATH) as f:
        baseline_summary = json.load(f)

    compare_keys = ["rouge1", "rouge2", "rougeL", "bleu", "perplexity_corpus",
                     "avg_generation_seconds_per_example"]
    comparison = pd.DataFrame({
        "baseline": {k: baseline_summary.get(k) for k in compare_keys},
        "lora": {k: lora_summary.get(k) for k in compare_keys},
    })
    comparison["delta"] = comparison["lora"] - comparison["baseline"]
    comparison["pct_change"] = (comparison["delta"] / comparison["baseline"].abs()) * 100
    display(comparison.round(4))
else:
    print("No baseline summary found — skipping comparison table. "
          "Set BASELINE_SUMMARY_PATH above and re-run this cell if you have it.")


In [ ]:
# Stratified by severity, same as baseline notebook, for the report's trade-off section
strat = df_test.groupby("Status").agg(
    n=("Doctor", "count"),
    rougeL=("rougeL", "mean"),
    perplexity=("perplexity", "mean"),
).round(4)
strat


In [ ]:
# Eyeball a few predictions vs references vs the baseline's style
for i in df_test.sample(3, random_state=SEED).index:
    row = df_test.loc[i]
    print("="*100)
    print("SEVERITY:", row["Status"])
    print("PATIENT :", row["Patient"][:250], "...")
    print("-"*100)
    print("REFERENCE DOCTOR:", row["Doctor"][:400])
    print("-"*100)
    print("LORA PREDICTION :", row["prediction"][:400])


## 9. Next steps

- `results/lora_summary.json` and `results/lora_predictions.csv` are saved for the final
  performance report.
- `lora_adapter/` holds the trained adapter (should be tens of MB, not hundreds) — confirm this
  before moving on.
- Remaining project-spec items: **Prompt Tuning** and **QLoRA** runs, using the identical
  evaluation pipeline above, to complete the three-way comparison table (ROUGE, PPL, latency,
  model size) for the final PDF report.
